In [1]:
#0. Étape 1. Imports et chemins

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from matplotlib import axis

from supply_chain_resilience.config import RAW_DATA_DIR

2026-10-20 15:28:10.066 | INFO     | supply_chain_resilience.config:<module>:11 - PROJ_ROOT path is: C:\Devs\python\Researchs\Article_4\supply_chain_resilience


In [2]:
#1. DataCo

dataco_path = RAW_DATA_DIR / "dataco" / "DataCoSupplyChainDataset.csv"

dataco_df = pd.read_csv(dataco_path, encoding="latin-1")

In [3]:
#1.1 Dataset

dataco_df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [4]:
#2. DataOrders to datetime

dataco_df["order date (DateOrders)"] = pd.to_datetime(dataco_df["order date (DateOrders)"], format="%m/%d/%Y %H:%M")

In [5]:
#3. DataOrders to datetime

dataco_df["shipping date (DateOrders)"] = pd.to_datetime(dataco_df["shipping date (DateOrders)"], format="%m/%d/%Y %H:%M")

In [6]:
#4. Dataset shape

dataco_df.shape

(180519, 53)

In [7]:
#. Min date

min_date = dataco_df["order date (DateOrders)"].min()

max_date = dataco_df["order date (DateOrders)"].max()

min_date, max_date

(Timestamp('2015-01-01 00:00:00'), Timestamp('2018-01-31 23:38:00'))

In [8]:
#5. Unique orders

n_orders = dataco_df["Order Id"].nunique()

n_orders

65752

In [9]:
rapport = len(dataco_df) / n_orders

rapport

2.745452609806546

In [10]:
#6. Cibles

dataco_df["delta"] = dataco_df["Days for shipping (real)"] - dataco_df["Days for shipment (scheduled)"]

dataco_df["y_clf"] = (dataco_df["delta"] > 0).astype(int)

dataco_df["y_reg"] = dataco_df["delta"].clip(lower=0)

In [11]:
#. Étape 4. Contrôle de la cible

lines_yes = dataco_df["y_clf"] == dataco_df["Late_delivery_risk"].astype(int)

pct_yes = lines_yes.mean() * 100

print(round(pct_yes, 2), "%")


97.55 %


In [12]:
#. Orders canceled

lines_no = dataco_df[~lines_yes]["Late_delivery_risk"]

n_no = lines_no.value_counts()

print(n_no)

Late_delivery_risk
0    4423
Name: count, dtype: int64


In [13]:
#. Canceled

dataco_df[~lines_yes]["Delivery Status"].value_counts()

Delivery Status
Shipping canceled    4423
Name: count, dtype: int64

In [14]:
no_cancel = dataco_df["Delivery Status"] != "Shipping canceled"

n_no_cancel = no_cancel.sum()

pct_without_cancel = lines_yes[no_cancel].mean() * 100

print(round(pct_without_cancel, 2), "%")

n_no_cancel

100.0 %


np.int64(172765)

In [15]:
#. Étape 5. Annulations et fraudes

status_crosstab = pd.crosstab(dataco_df["Order Status"], dataco_df["Delivery Status"])

status_crosstab


Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Order Status,,,,
CANCELED,0,0,3692,0
CLOSED,4809,11109,0,3698
COMPLETE,14136,34199,0,11156
ON_HOLD,2413,5450,0,1941
PAYMENT_REVIEW,420,1082,0,391
PENDING,4857,11712,0,3658
PENDING_PAYMENT,9588,22922,0,7322
PROCESSING,5369,12503,0,4030
SUSPECTED_FRAUD,0,0,4062,0


In [16]:
#. Étape 6. Constance dans une commande

per_order = dataco_df.groupby("Order Id")[["y_clf", "Shipping Mode", "order date (DateOrders)", "Delivery Status"]].nunique()

(per_order > 1).sum()

y_clf                      0
Shipping Mode              0
order date (DateOrders)    0
Delivery Status            0
dtype: int64

In [17]:
#. Étape 7. Structure du délai

pd.crosstab(dataco_df["Shipping Mode"], dataco_df["Days for shipment (scheduled)"])

Days for shipment (scheduled),0,1,2,4
Shipping Mode,,,,
First Class,0,27814,0,0
Same Day,9737,0,0,0
Second Class,0,0,35216,0
Standard Class,0,0,0,107752


In [18]:
dataco_df["delta"].value_counts().sort_index()

delta
-2    21666
-1    21700
 0    33753
 1    60647
 2    28718
 3     7052
 4     6983
Name: count, dtype: int64

In [19]:
dataco_df[no_cancel].groupby("Shipping Mode")["y_clf"].mean()

Shipping Mode
First Class       1.000000
Same Day          0.479285
Second Class      0.798290
Standard Class    0.397691
Name: y_clf, dtype: float64

In [20]:
#. Étape 8. Prévalence par trimestre

without_canceled = dataco_df[no_cancel].drop_duplicates("Order Id")

without_canceled.shape

(62897, 56)

In [21]:
display(without_canceled.groupby(without_canceled["order date (DateOrders)"].dt.to_period("Q"))["y_clf"].agg(["size", "mean"]))

pct_without_canceled = without_canceled["y_clf"].mean() * 100

print(round(pct_without_canceled, 2), "%")

,size,mean
order date (DateOrders),,
2015Q1,4920,0.571545
2015Q2,4974,0.570366
2015Q3,5016,0.574163
2015Q4,5085,0.570698
2016Q1,4977,0.578260
2016Q2,4922,0.581877
2016Q3,5024,0.568073
2016Q4,5032,0.576510
2017Q1,4933,0.568214


57.31 %


In [22]:
#. Étape 9. Rupture de fin de période

monthly_orders = dataco_df.groupby(dataco_df["order date (DateOrders)"].dt.to_period("M")).agg(lignes=("Order Id", "size"), commandes=("Order Id", "nunique"), produits=("Product Name", "nunique"))

monthly_orders.loc["2017-06":]

,lignes,commandes,produits
order date (DateOrders),,,
2017-06,4951,1676,42
2017-07,5318,1776,42
2017-08,5305,1768,42
2017-09,5189,1723,53
2017-10,2255,2101,20
2017-11,2055,2055,8
2017-12,2124,2124,14
2018-01,2123,2123,10


In [23]:
monthly_orders["lignes"] / monthly_orders["commandes"]

order date (DateOrders)
2015-01    2.978176
2015-02    2.983596
2015-03    3.010668
2015-04    2.997661
2015-05    3.016329
2015-06    2.976232
2015-07    3.005672
2015-08    2.992622
2015-09    3.012896
2015-10    2.987042
2015-11    3.027762
2015-12    2.975623
2016-01    3.000564
2016-02    2.966061
2016-03    2.953515
2016-04    2.987691
2016-05    3.007374
2016-06    2.990533
2016-07    3.017634
2016-08    3.020385
2016-09    2.982659
2016-10    3.039414
2016-11    3.027310
2016-12    2.988656
2017-01    2.989685
2017-02    3.039653
2017-03    3.000561
2017-04    2.997125
2017-05    3.015882
2017-06    2.954057
2017-07    2.994369
2017-08    3.000566
2017-09    3.011608
2017-10    1.073298
2017-11    1.000000
2017-12    1.000000
2018-01    1.000000
Freq: M, dtype: float64

In [24]:
#. Étape 10. Qualité des données
#. a. Part de valeurs manquantes par colonne : .isna().mean().

dataco_df.isna().mean()

Type                             0.000000
Days for shipping (real)         0.000000
Days for shipment (scheduled)    0.000000
Benefit per order                0.000000
Sales per customer               0.000000
Delivery Status                  0.000000
Late_delivery_risk               0.000000
Category Id                      0.000000
Category Name                    0.000000
Customer City                    0.000000
Customer Country                 0.000000
Customer Email                   0.000000
Customer Fname                   0.000000
Customer Id                      0.000000
Customer Lname                   0.000044
Customer Password                0.000000
Customer Segment                 0.000000
Customer State                   0.000000
Customer Street                  0.000000
Customer Zipcode                 0.000017
Department Id                    0.000000
Department Name                  0.000000
Latitude                         0.000000
Longitude                        0

In [25]:
#. b. Colonnes constantes : celles où .nunique(dropna=False) est inférieur ou égal à 1.

dataco_df.nunique(dropna=False) <= 1

Type                             False
Days for shipping (real)         False
Days for shipment (scheduled)    False
Benefit per order                False
Sales per customer               False
Delivery Status                  False
Late_delivery_risk               False
Category Id                      False
Category Name                    False
Customer City                    False
Customer Country                 False
Customer Email                    True
Customer Fname                   False
Customer Id                      False
Customer Lname                   False
Customer Password                 True
Customer Segment                 False
Customer State                   False
Customer Street                  False
Customer Zipcode                 False
Department Id                    False
Department Name                  False
Latitude                         False
Longitude                        False
Market                           False
Order City               

In [26]:
#.  c. Doublons, en ignorant Order Item Id : .duplicated().sum().

subset_cols = [col for col in dataco_df.columns if col not in ["Order Item Id"]]

dataco_df.duplicated(subset=subset_cols).sum()

np.int64(0)

In [27]:
#. d. Colonnes en double

pairs = [
    ("Order Item Cardprod Id", "Product Card Id"),
    ("Order Customer Id", "Customer Id"),
    ("Benefit per order", "Order Profit Per Order"),
    ("Sales per customer", "Order Item Total"),
    ("Order Item Product Price", "Product Price"),
    ("Product Category Id", "Category Id"),
]

for a, b in pairs:
    print(a, "==", b, ":", (dataco_df[a] == dataco_df[b]).all())

Order Item Cardprod Id == Product Card Id : True
Order Customer Id == Customer Id : True
Benefit per order == Order Profit Per Order : True
Sales per customer == Order Item Total : True
Order Item Product Price == Product Price : True
Product Category Id == Category Id : True


In [28]:
#. e. Fuite de la date d'expédition

ship_day = dataco_df["shipping date (DateOrders)"].dt.normalize()

order_day = dataco_df["order date (DateOrders)"].dt.normalize()

gap_days = (ship_day - order_day).dt.days

(gap_days == dataco_df["Days for shipping (real)"]).mean()

np.float64(1.0)

### Constats de l'étape 10 : colonnes de DataCo à supprimer en phase 3

Aucune ligne en double (en ignorant `Order Item Id`).

**1. Vides ou constantes : aucune information**

| Colonne | Preuve |
|---|---|
| `Product Description` | 100 % de valeurs manquantes |
| `Product Status` | une seule valeur (0) |
| `Customer Email` | une seule valeur, masquée (`XXXXXXXXX`) |
| `Customer Password` | une seule valeur, masquée (`XXXXXXXXX`) |

**2. Trop incomplète**

| Colonne | Preuve |
|---|---|
| `Order Zipcode` | 86,2 % de valeurs manquantes |

**3. Données personnelles : inutiles pour prédire et sensibles**

| Colonne | Contenu |
|---|---|
| `Customer Fname`, `Customer Lname` | prénom et nom du client |
| `Customer Street` | adresse du client |

On garde `Customer City`, `Customer State` et `Customer Zipcode` comme variables géographiques. `Customer Zipcode` a 3 valeurs manquantes, à traiter en phase 3.

**4. Fuite de la cible : connues seulement après la livraison**

| Colonne | Pourquoi elle fuit |
|---|---|
| `Days for shipping (real)` | sert à calculer `delta` |
| `shipping date (DateOrders)` | l'écart en jours calendaires avec la date de commande redonne le délai réel sur 100 % des lignes (étape 10e) |
| `Delivery Status` | dit directement si la livraison est en retard |
| `Late_delivery_risk` | égale à `y_clf` sur 100 % des lignes non annulées (étape 4) |
| `Order Status` | état de la commande après coup ; `CANCELED` et `SUSPECTED_FRAUD` désignent exactement les annulations (étape 5) |

**5. En double : on garde une colonne par paire (étape 10d)**

| On supprime | Copie exacte de |
|---|---|
| `Order Item Cardprod Id` | `Product Card Id` |
| `Order Customer Id` | `Customer Id` |
| `Benefit per order` | `Order Profit Per Order` |
| `Sales per customer` | `Order Item Total` |
| `Order Item Product Price` | `Product Price` |
| `Product Category Id` | `Category Id` |

**Identifiants : gardés, mais hors du modèle**

`Order Id` (partition groupée par commande), `Order Item Id`, `Customer Id`.

In [29]:
#. Partie 2. Olist
#. Étape 11. Chargement et statuts

olist_path = RAW_DATA_DIR / "olist" / "olist_orders_dataset.csv"

olist_df = pd.read_csv(olist_path, encoding="utf-8")

olist_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [30]:
#. 2. Conversion des dates

olist_df["order_purchase_timestamp"] = pd.to_datetime(olist_df["order_purchase_timestamp"])
olist_df["order_approved_at"] = pd.to_datetime(olist_df["order_approved_at"])
olist_df["order_delivered_carrier_date"] = pd.to_datetime(olist_df["order_delivered_carrier_date"])
olist_df["order_delivered_customer_date"] = pd.to_datetime(olist_df["order_delivered_customer_date"])
olist_df["order_estimated_delivery_date"] = pd.to_datetime(olist_df["order_estimated_delivery_date"])

In [31]:
#. 3. Forme

olist_df.shape

(99441, 8)

In [32]:
#. Clé

olist_df.order_id.nunique()

99441

In [33]:
#.  Période

olist_df.order_purchase_timestamp.min(), olist_df.order_purchase_timestamp.max()

(Timestamp('2016-09-04 21:15:19'), Timestamp('2018-10-17 17:30:18'))

In [34]:
#. 5. Statuts

olist_df.order_status.value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [35]:
#. 6. Valeurs manquantes

olist_df.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [36]:
#. 7. Livrées sans date

cmd_delivered = olist_df["order_status"] == "delivered"

cmd_without_date = olist_df.order_delivered_customer_date.isna()

cmd_delivered.sum(), (cmd_delivered & cmd_without_date).sum()

(np.int64(96478), np.int64(8))

In [37]:
#. Étape 12. Olist : la cible

#. 1. Filtre

olist_delivered = olist_df[cmd_delivered & ~cmd_without_date].copy()

olist_delivered.shape

(96470, 8)

In [38]:
#. delta en jours calendaires

olist_delivered["delta"] = (olist_delivered["order_delivered_customer_date"].dt.normalize() - olist_delivered["order_estimated_delivery_date"]).dt.days

In [39]:
#. Cibles, comme pour DataCo

olist_delivered["y_clf"] = (olist_delivered["delta"] > 0).astype(int)

olist_delivered["y_reg"] = olist_delivered["delta"].clip(lower=0)

In [40]:
#. 4. Prévalence

olist_delivered["y_clf"].mean(), olist_delivered["y_clf"].sum()

(np.float64(0.06773090079817559), np.int64(6534))

In [41]:
#. 5. Distribution de delta

olist_delivered["delta"].describe()

count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delta, dtype: float64

In [42]:
#. 6. Ampleur des retards

mask = olist_delivered["y_clf"] == 1

olist_delivered[mask]["y_reg"].describe()

count    6534.000000
mean       10.620141
std        14.644955
min         1.000000
25%         3.000000
50%         7.000000
75%        13.000000
max       188.000000
Name: y_reg, dtype: float64

In [43]:
#. 7. Prévalence par année d'achat

olist_delivered.groupby(olist_delivered["order_purchase_timestamp"].dt.year)["y_clf"].agg(["size", "mean"])

,size,mean
order_purchase_timestamp,,
2016,267,0.011236
2017,43426,0.056487
2018,52777,0.077269


In [44]:
#. Étape 13. Olist : exclusion de 2016 et troncature de fin de période

#. 1. Exclusion de 2016

over_2016 = olist_delivered["order_purchase_timestamp"].dt.year >= 2017

olist_2017 = olist_delivered[over_2016].copy()

olist_2017.shape

(96203, 11)

In [45]:
#. 2. Preuve de la censure

olist_2017["delivery_days"] = (olist_2017["order_delivered_customer_date"] - olist_2017["order_purchase_timestamp"]).dt.days

weekly_delay = olist_2017.groupby(olist_2017["order_purchase_timestamp"].dt.to_period("W"))["delivery_days"].mean()

weekly_delay.tail(8)

order_purchase_timestamp
2018-07-09/2018-07-15    10.797325
2018-07-16/2018-07-22     8.574287
2018-07-23/2018-07-29     7.867081
2018-07-30/2018-08-05     8.222777
2018-08-06/2018-08-12     8.035733
2018-08-13/2018-08-19     7.181620
2018-08-20/2018-08-26     5.038899
2018-08-27/2018-09-02     1.793103
Freq: W-SUN, Name: delivery_days, dtype: float64

In [46]:
#. 3. Fin d'observation : dernière approbation de commande (table complète)

end_obs = olist_df["order_approved_at"].max()

#. 4. Horizon : 95e centile de la durée de livraison

horizon = olist_2017["delivery_days"].quantile(0.95)

#. 5. Date de coupure

cutoff = end_obs.normalize() - pd.Timedelta(days=horizon)

end_obs, horizon, cutoff

(Timestamp('2018-09-03 17:40:06'),
 np.float64(29.0),
 Timestamp('2018-08-05 00:00:00'))

In [47]:
#. 6. Troncature

censored = olist_2017["order_purchase_timestamp"] >= cutoff

olist_kept = olist_2017[~censored].copy()

#. 7. Prévalence finale

censored.sum(), len(olist_kept), olist_kept["y_clf"].mean()

(np.int64(5209), 90994, np.float64(0.06867485768292415))

In [48]:
#. Étape 14. Olist : dérive de la prévalence et coupure chronologique

#. 1. Prévalence par trimestre

olist_kept.groupby(olist_kept["order_purchase_timestamp"].dt.to_period("Q"))["y_clf"].agg(["size", "mean"])

,size,mean
order_purchase_timestamp,,
2017Q1,4949,0.037785
2017Q2,8983,0.039185
2017Q3,12215,0.033729
2017Q4,17279,0.086926
2018Q1,20627,0.128812
2018Q2,19643,0.041745
2018Q3,7298,0.043711


In [49]:
#. 2. Coupure chronologique : les 25 % de commandes les plus récentes en test (provisoire)

chrono_cut = olist_kept["order_purchase_timestamp"].quantile(0.75)

print(chrono_cut)

#. 3. Entraînement (False) et test (True)

is_test = olist_kept["order_purchase_timestamp"] >= chrono_cut

olist_kept.groupby(is_test)["y_clf"].agg(["size", "mean"])

2018-04-19 12:45:51.750000


,size,mean
order_purchase_timestamp,,
False,68245,0.078116
True,22749,0.040353


In [50]:
#. Étape 15. DataCo : coupure chronologique et rupture dans le test

#. 1. Coupure : les 25 % de commandes les plus récentes en test (provisoire)

dataco_cut = without_canceled["order date (DateOrders)"].quantile(0.75)

print(dataco_cut)

#. 2. Entraînement (False) et test (True)

dataco_is_test = without_canceled["order date (DateOrders)"] >= dataco_cut

without_canceled.groupby(dataco_is_test)["y_clf"].agg(["size", "mean"])

2017-05-12 10:42:00


,size,mean
order date (DateOrders),,
False,47172,0.572522
True,15725,0.574944


In [51]:
#. 3. Part du test dans le nouveau régime (à partir d'octobre 2017, étape 9)

new_regime = without_canceled["order date (DateOrders)"] >= "2017-10-01"

(dataco_is_test & new_regime).sum(), dataco_is_test.sum()

(np.int64(8043), np.int64(15725))

In [52]:
#. Étape 16. Tableau de comparaison des deux jeux (chiffres provisoires, recalculés en phase 3)

dataco_date = without_canceled["order date (DateOrders)"]
olist_date = olist_kept["order_purchase_timestamp"]

dataco_quarterly = without_canceled.groupby(dataco_date.dt.to_period("Q"))["y_clf"].mean()
olist_quarterly = olist_kept.groupby(olist_date.dt.to_period("Q"))["y_clf"].mean()

dataco_late = without_canceled.loc[without_canceled["y_clf"] == 1, "y_reg"]
olist_late = olist_kept.loc[olist_kept["y_clf"] == 1, "y_reg"]

pct = lambda x: f"{x:.1%}"

comparison = pd.DataFrame(
    {
        "DataCo": [
            f"ligne d'article ({rapport:.2f} par commande)",
            len(without_canceled),
            f"{dataco_date.min():%Y-%m} à {dataco_date.max():%Y-%m}",
            pct(without_canceled["y_clf"].mean()),
            f"{pct(without_canceled.loc[~dataco_is_test, 'y_clf'].mean())} / {pct(without_canceled.loc[dataco_is_test, 'y_clf'].mean())}",
            f"{pct(dataco_quarterly.min())} à {pct(dataco_quarterly.max())}",
            f"{dataco_cut:%Y-%m-%d}",
            "fixé par le mode d'expédition (0, 1, 2 ou 4 jours)",
            f"{dataco_late.median():.0f} / {dataco_late.mean():.1f}",
        ],
        "Olist": [
            "commande",
            len(olist_kept),
            f"{olist_date.min():%Y-%m} à {olist_date.max():%Y-%m}",
            pct(olist_kept["y_clf"].mean()),
            f"{pct(olist_kept.loc[~is_test, 'y_clf'].mean())} / {pct(olist_kept.loc[is_test, 'y_clf'].mean())}",
            f"{pct(olist_quarterly.min())} à {pct(olist_quarterly.max())}",
            f"{chrono_cut:%Y-%m-%d}",
            "date estimée, propre à chaque commande",
            f"{olist_late.median():.0f} / {olist_late.mean():.1f}",
        ],
    },
    index=[
        "Unité d'observation",
        "Commandes gardées",
        "Période d'achat",
        "Prévalence",
        "Prévalence entraînement / test",
        "Prévalence par trimestre",
        "Coupure chronologique",
        "Délai prévu",
        "Retard : médiane / moyenne (jours)",
    ],
)

comparison

,DataCo,Olist
Unité d'observation,ligne d'article (2.75 par commande),commande
Commandes gardées,62897,90994
Période d'achat,2015-01 à 2018-01,2017-01 à 2018-08
Prévalence,57.3%,6.9%
Prévalence entraînement / test,57.3% / 57.5%,7.8% / 4.0%
Prévalence par trimestre,56.5% à 58.7%,3.4% à 12.9%
Coupure chronologique,2017-05-12,2018-04-19
Délai prévu,"fixé par le mode d'expédition (0, 1, 2 ou 4 jo...","date estimée, propre à chaque commande"
Retard : médiane / moyenne (jours),1 / 1.6,7 / 10.9


### Constats de l'étape 17 : cibles, périodes et dérive (chiffres provisoires, recalculés en phase 3)

**1. Olist : données retenues (étapes 11 à 13)**

- Commandes au statut `delivered` avec une date de livraison : 96 470.
- Achats de 2016 exclus (267 commandes) : il en reste 96 203.
- Après troncature de fin de période : **90 994 commandes**, achetées de janvier 2017 au 04/08/2018. **Prévalence 6,87 %.**

**2. Censure de fin de période (étape 13)**

- Seules les commandes déjà livrées figurent dans l'extraction : les dernières semaines ne gardent que les livraisons rapides. Le délai moyen par semaine d'achat, d'environ 8 jours, tombe à 5,0 puis 1,8 jours sur les deux dernières semaines.
- Règle retenue : fin d'observation = dernière approbation de commande (03/09/2018) ; on exclut les achats des 29 derniers jours (95e centile du délai de livraison), soit à partir du 05/08/2018 : **5 209 commandes exclues**.

**3. Dérive entre entraînement et test (étapes 8, 9, 14 et 15)**

Coupure chronologique provisoire : les 25 % de commandes les plus récentes en test.

| | DataCo | Olist |
|---|---|---|
| Coupure | 12/05/2017 | 19/04/2018 |
| Prévalence par trimestre | 56,5 % à 58,7 % | 3,4 % à 12,9 % |
| Prévalence entraînement / test | 57,3 % / 57,5 % | 7,8 % / 4,0 % |
| Nature de la dérive | décalage des covariables, sans dérive de prévalence | décalage de prévalence |

- **Olist** : la prévalence monte en 2017Q4 (8,7 %) et 2018Q1 (12,9 %), puis redescend vers 4 %. La prévalence du test vaut environ la moitié de celle de l'entraînement : à rapporter avec les résultats.
- **DataCo** : à partir d'octobre 2017, 1 ligne par commande au lieu de 3, et 8 à 20 produits par mois au lieu de 42 à 53. **8 043 des 15 725 commandes du test (51 %)** sont dans ce nouveau régime. À décrire au §5.1.

**4. Ampleur des retards, sur les commandes gardées (étape 16)**

- **DataCo** : retard de 1 à 4 jours, médiane 1, moyenne 1,6. Le délai prévu est fixé par le mode d'expédition (0, 1, 2 ou 4 jours), et First Class est toujours en retard : indice de données générées.
- **Olist** : 6 249 retards, longue traîne, médiane 7, moyenne 10,9, maximum 188 jours. La date estimée est propre à chaque commande.
- Conséquence : la régression est bien plus difficile sur Olist, et la métrique de régression doit résister aux valeurs extrêmes.